In [1]:
import os

def _parse_xlim_env(name):
    raw = os.environ.get(name)
    if not raw:
        return None
    a, b = raw.split(',')
    return (float(a), float(b))

exp_save_path = os.environ.get('EXP_SAVE_PATH', r'C:\Users\rehan\OneDrive\00_TO-DO\paper_review\Masterarbeit_Butt\Paper_abgabe\simulation_study\results\(n_train)1000__(B_RF)1000__(B_1)200__(n_sim)1000__(seed)42__4kovariates_test_boot\121.901767')
patient = os.environ.get('PATIENT_LABEL', 'averageS')

corr_xlim_map = {
    1: _parse_xlim_env('CORR_XLIM_1'),
    3: _parse_xlim_env('CORR_XLIM_3'),
    5: _parse_xlim_env('CORR_XLIM_5'),
}


In [2]:
import pandas as pd
import json
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os

# erstelle verzeichnis, falls nicht vorhanden /help_figures
if not os.path.exists('help_figures'):
    os.makedirs('help_figures')

with open(exp_save_path + '/setting.json') as f:
    exp_settings = json.load(f)
S_t = exp_settings["true_survival_probability[1,3,5]"]

# lade results
results1 = pd.read_csv(exp_save_path + f"/results1__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][0]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][0]}.csv")
results3 = pd.read_csv(exp_save_path + f"/results3__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][1]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][1]}.csv")
results5 = pd.read_csv(exp_save_path + f"/results5__(zero_weights){exp_settings['portion_zero_weights_train[1,3,5]'][2]}__(seen_events){exp_settings['portion_seen_events_train[1,3,5]'][2]}.csv")
x_pred = exp_settings['X_pred_point']

#### plot corrs

In [3]:
def plot_corr(results1, prop_weights, patient, xlim=None):
    boot_var  = results1['boot_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)
    ijk_u_jahn = results1['ijk_u_jahn_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)    
    ijk_u_wager  = results1['ijk_u_wager_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)

    if patient == 'averageS':
        df = pd.DataFrame({ r"$\hat{\sigma}_{IJ-U}^{wB} \left(\hat{\theta}_{\tau}(x_{0})\right)$":   ijk_u_jahn, 
                            r"$\hat{\sigma}_{IJ-U}^{B} \left(\hat{\theta}_{\tau}(x_{0})\right)$": ijk_u_wager, 
                             r"$\hat{\sigma}_{Boot} \left(\hat{\theta}_{\tau}(x_{0})\right)$":   boot_var})
    elif patient == 'higherS':
        df = pd.DataFrame({ r"$\hat{\sigma}_{IJ-U}^{wB} \left(\hat{\theta}_{\tau}(x_{high})\right)$":   ijk_u_jahn, 
                            r"$\hat{\sigma}_{IJ-U}^{B} \left(\hat{\theta}_{\tau}(x_{high})\right)$": ijk_u_wager, 
                             r"$\hat{\sigma}_{Boot} \left(\hat{\theta}_{\tau}(x_{high})\right)$":   boot_var})
    elif patient == 'lowerS':
        df = pd.DataFrame({ r"$\hat{\sigma}_{IJ-U}^{wB} \left(\hat{\theta}_{\tau}(x_{low})\right)$":   ijk_u_jahn, 
                            r"$\hat{\sigma}_{IJ-U}^{B} \left(\hat{\theta}_{\tau}(x_{low})\right)$": ijk_u_wager, 
                             r"$\hat{\sigma}_{Boot} \left(\hat{\theta}_{\tau}(x_{low})\right)$":   boot_var})

    # Gemeinsame Achsenskala berechnen
    if xlim is not None:
        axis_min, axis_max = float(xlim[0]), float(xlim[1])
    else:
        global_min = df.min().min()
        global_max = df.max().max()
        margin = 0.05 * (global_max - global_min)
        axis_min = global_min - margin
        axis_max = global_max + margin

    # Plot
    g = sns.PairGrid(df, diag_sharey=False, height=2.1, aspect=1.0)
    g.map_lower(sns.scatterplot, alpha=0.5)

    def plot_line_x_equals_y(x, y, **kwargs):
        ax = plt.gca()
        ax.plot([axis_min, axis_max], [axis_min, axis_max],
                color='red', linestyle='--', linewidth=1)

    g.map_lower(plot_line_x_equals_y)
    g.map_diag(sns.histplot, kde=True, color="gray")

    # Obere Dreieck-Achsen ausblenden
    for i, j in zip(*np.triu_indices_from(g.axes, 1)):
        g.axes[i, j].set_visible(False)

    # Achsengleichheit setzen
    for i in range(len(df.columns)):
        for j in range(len(df.columns)):
            ax = g.axes[i, j]
            if i != j:
                ax.set_xlim(axis_min, axis_max)
                ax.set_ylim(axis_min, axis_max)

    # Achsentitel etc.
    for i, colname in enumerate(df.columns):
        for j in range(len(df.columns)):
            if i == j:
                ax = g.axes[j, i]
                ax.set_title(colname, fontsize=12, fontweight='bold')

    for ax_row in g.axes:
        for ax in ax_row:
            ax.set_xlabel("")
            ax.set_ylabel("")

    g.fig.tight_layout()
    plt.savefig(f"./help_figures/corr{prop_weights}{patient}.jpeg", dpi=300)
    plt.close()

plot_corr(results1, 1, patient, xlim=corr_xlim_map[1])
plot_corr(results3, 3, patient, xlim=corr_xlim_map[3])
plot_corr(results5, 5, patient, xlim=corr_xlim_map[5])
